# Supervised Learning — Parts 5 to 13
## Spam / Phishing Email Detection — CEAS_08 Dataset

## 0. Setup

### 0.1 Imports and configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                     train_test_split)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)

RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5
MAX_TFIDF_FEATURES = 300
NGRAM_RANGE = (1, 2)

DATA_DIR = Path("artifacts")
CLEAN_DATA = DATA_DIR / "ceas08_clean.parquet"

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

### 0.2 Load the cleaned dataset

In [2]:
if not CLEAN_DATA.exists():
    raise FileNotFoundError(
        f"{CLEAN_DATA} not found. Run Final_EDA.ipynb first to generate it."
    )

df = pd.read_parquet(CLEAN_DATA)
print("Shape:", df.shape)
df.head(3)

Shape: (39154, 13)


,sender,receiver,date,subject,body,label,urls,date_parsed,date_suspicious,sender_domain_raw,sender_domain,body_len,subject_len
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1,2008-08-05 23:31:02+00:00,False,iworld.de,iworld.de,273,25
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1,2008-08-05 23:31:03+00:00,False,icable.ph,icable.ph,82,22
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1,2008-08-06 08:28:00+00:00,False,universalnet.psi.br,universalnet.psi.br,3918,20


### 0.3 Sanity checks

In [3]:
print(df.dtypes.to_string())
print()
print("Missing values:", df.isna().sum().sum())
print("Class balance:")
print((df["label"].value_counts(normalize=True) * 100).round(2).to_string())

sender                               str
receiver                             str
date                                 str
subject                              str
body                                 str
label                              int64
urls                               int64
date_parsed          datetime64[us, UTC]
date_suspicious                     bool
sender_domain_raw                    str
sender_domain                        str
body_len                           int64
subject_len                        int64

Missing values: 1925
Class balance:
label
1    55.78
0    44.22


## Part 5 — Feature Preparation

### 5.1 Derived features

In [ ]:
df["text"] = df["subject"] + " " + df["body"]
df["log_body_len"] = np.log1p(df["body_len"])

df[["text", "log_body_len", "subject_len"]].head(3)

### 5.2 Data leakage review

In [ ]:
sender_freq = df["sender"].map(df["sender"].value_counts())
receiver_ratio = df["receiver"].map(df.groupby("receiver")["label"].mean())

pd.DataFrame({
    "corr_with_label": [sender_freq.corr(df["label"]),
                        receiver_ratio.corr(df["label"])],
    "pure_single_class_groups": [
        f'{df.groupby("sender")["label"].nunique().eq(1).sum()} / {df["sender"].nunique()}',
        f'{df.groupby("receiver")["label"].nunique().eq(1).sum()} / {df["receiver"].nunique()}',
    ],
}, index=["sender_frequency", "receiver_spam_ratio"]).round(4)

In [ ]:
pd.DataFrame({
    "spam_rate_%": [df.loc[sender_freq == 1, "label"].mean() * 100,
                    df.loc[sender_freq > 1, "label"].mean() * 100],
    "n_rows": [(sender_freq == 1).sum(), (sender_freq > 1).sum()],
}, index=["sender seen once", "sender seen 2+ times"]).round(1)

### 5.3 Feature groups

In [ ]:
TARGET = "label"
TEXT_COL = "text"
NUMERIC_COLS = ["urls", "log_body_len", "subject_len", "date_suspicious"]

DROPPED_COLS = ["sender", "receiver", "sender_domain", "sender_domain_raw",
                "date", "date_parsed", "subject", "body", "body_len"]

FEATURE_COLS = [TEXT_COL] + NUMERIC_COLS

print("Kept   :", FEATURE_COLS)
print("Dropped:", DROPPED_COLS)

### 5.4 Preprocessing pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=MAX_TFIDF_FEATURES,
                                 ngram_range=NGRAM_RANGE,
                                 stop_words="english"), TEXT_COL),
        ("numeric", MinMaxScaler(), NUMERIC_COLS),
    ]
)

preprocessor

### 5.5 Resulting feature space

In [ ]:
_probe = preprocessor.fit_transform(df[FEATURE_COLS])

print("Feature matrix:", _probe.shape)
print("Sparse:", hasattr(_probe, "toarray"))
print("Min value:", _probe.min(), "(non-negative keeps MultinomialNB valid)")
print("Missing values in features:", df[FEATURE_COLS].isna().sum().sum())

del _probe

## Part 6 — Data Splitting

### 6.1 Train / test split

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]:,} rows ({(1 - TEST_SIZE):.0%})")
print(f"Test : {X_test.shape[0]:,} rows ({TEST_SIZE:.0%})")

### 6.2 Stratification check

In [ ]:
pd.DataFrame({
    "full": y.value_counts(normalize=True) * 100,
    "train": y_train.value_counts(normalize=True) * 100,
    "test": y_test.value_counts(normalize=True) * 100,
}).round(2)

### 6.3 Cross-validation strategy

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc"]

cv

### 6.4 Leakage prevention

In [ ]:
def make_pipeline(estimator):
    """Fresh preprocessor per model so TF-IDF/scaler are fit on training folds only."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("text", TfidfVectorizer(max_features=MAX_TFIDF_FEATURES,
                                     ngram_range=NGRAM_RANGE,
                                     stop_words="english"), TEXT_COL),
            ("numeric", MinMaxScaler(), NUMERIC_COLS),
        ])),
        ("model", estimator),
    ])


make_pipeline(MultinomialNB())

### 6.5 Shared evaluation helper

In [ ]:
results = []


def evaluate(name, estimator, cross_validate_too=True):
    pipe = make_pipeline(estimator)
    pipe.fit(X_train, y_train)

    pred_train = pipe.predict(X_train)
    pred_test = pipe.predict(X_test)
    proba_test = (pipe.predict_proba(X_test)[:, 1]
                  if hasattr(pipe["model"], "predict_proba") else None)

    row = {
        "model": name,
        "train_acc": accuracy_score(y_train, pred_train),
        "test_acc": accuracy_score(y_test, pred_test),
        "precision": precision_score(y_test, pred_test, average="macro", zero_division=0),
        "recall": recall_score(y_test, pred_test, average="macro"),
        "f1": f1_score(y_test, pred_test, average="macro"),
        "roc_auc": roc_auc_score(y_test, proba_test) if proba_test is not None else np.nan,
    }

    if cross_validate_too:
        cv_res = cross_validate(pipe, X_train, y_train, cv=cv,
                                scoring="f1_macro", n_jobs=-1)
        row["cv_f1_mean"] = cv_res["test_score"].mean()
        row["cv_f1_std"] = cv_res["test_score"].std()

    results.append(row)
    return pipe, row

## Part 7 — Baseline Model

### 7.1 Trivial reference — majority class

In [ ]:
dummy_pipe, dummy_row = evaluate(
    "Dummy (majority class)",
    DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
)

pd.Series(dummy_row).to_frame("value").round(4)

### 7.2 Baseline — Multinomial Naive Bayes

In [ ]:
nb_pipe, nb_row = evaluate("MultinomialNB (baseline)", MultinomialNB())

pd.Series(nb_row).to_frame("value").round(4)

### 7.3 Classification report and confusion matrix

In [ ]:
nb_pred = nb_pipe.predict(X_test)

print(classification_report(y_test, nb_pred,
                            target_names=["Legitimate (0)", "Spam/Phishing (1)"]))

cm = confusion_matrix(y_test, nb_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Legitimate", "Spam"], yticklabels=["Legitimate", "Spam"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("MultinomialNB — Confusion Matrix")
plt.tight_layout()
plt.show()

### 7.4 Underfitting check — train vs test vs cross-validation

In [ ]:
baseline = pd.DataFrame(results).set_index("model")
baseline["train_test_gap"] = baseline["train_acc"] - baseline["test_acc"]

baseline.round(4)

### 7.5 Score to beat

In [ ]:
BASELINE_F1 = nb_row["f1"]
BASELINE_ACC = nb_row["test_acc"]

print(f"Baseline macro F1 : {BASELINE_F1:.4f}")
print(f"Baseline accuracy : {BASELINE_ACC:.4f}")
print(f"Majority-class floor: {dummy_row['test_acc']:.4f}")

## Part 8 — Machine Learning Model Development

### 8.1 Model 1

### 8.2 Model 2

### 8.3 Model 3

### 8.4 Model 4

### 8.5 Model comparison table

## Part 9 — Model Evaluation

### 9.1 Metrics for every model

### 9.2 Confusion matrices

### 9.3 ROC and Precision-Recall curves

### 9.4 Comparison table

## Part 10 — Overfitting and Underfitting Analysis

### 10.1 Train vs validation performance

### 10.2 Learning curves

### 10.3 Diagnosis per model

## Part 11 — Hyperparameter Experiments

### 11.1 Experiment 1

### 11.2 Experiment 2

### 11.3 Experiment 3

### 11.4 Summary

## Part 12 — Model Comparison and Selection

### 12.1 Final comparison table

### 12.2 Final model selection

### 12.3 Held-out test set evaluation

## Part 13 — Error Analysis

### 13.1 Confusion matrix of the final model

### 13.2 False positives

### 13.3 False negatives

### 13.4 Error patterns